<a href="https://colab.research.google.com/github/zombimann/Mathematical-video-animations-and-visualization/blob/main/Pan_African_icons_voronoi_stippling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Voronoi Stippling Animation: Kimathi, Lumumba, Marley & Speed

This notebook implements a high-fidelity algorithmic animation of four Pan-African and cultural icons using **Weighted Voronoi Stippling**. The process transforms standard grayscale images into dynamic stippled portraits where point density and size reflect the underlying image intensity.

### Featured Icons
*   **Dedan Kimathi**: A key leader of the Mau Mau Uprising against British colonial rule in Kenya, symbolizing the struggle for land and freedom.
*   **Patrice Lumumba**: The first Prime Minister of the independent Republic of the Congo and a prominent pan-Africanist who stood against Belgian imperialism.
*   **Bob Marley**: A global musical ambassador for Reggae and a cultural icon whose work promoted peace, unity, and African liberation.
*   **IShowSpeed**: A modern digital icon whose explosive energy and global reach represent the new age of digital influence and connectivity.

### Technical Workflow
1.  **Importance Mapping**: Grayscale images are inverted and normalized. A contrast power factor ($p=1.8$) is applied to create a probability distribution that favors darker regions.
2.  **Lloyd's Relaxation**: A vectorized implementation using `scipy.spatial.cKDTree` and `np.bincount`. Over 40 iterations, 4,000 stipple points are iteratively moved toward the centroids of their Voronoi cells, weighted by the local image density.
3.  **Dynamic Rendering**: The animation features smooth transitions where points grow from 500 to 4,000 and migrate to their relaxed positions. Stipple radii are dynamically calculated (1.0 to 3.5 units) based on local importance.
4.  **Video Synthesis**: An H.264 encoded MP4 video (720p equivalent scale) is produced at 30 FPS, including title cards, descriptive lower-thirds, and a persistent semi-transparent watermark.

### Usage Note: Image Assets
The script fetches images from external web hosts. Due to the ephemeral nature of web links and anti-scraping policies (403/429 errors), you may occasionally need to update the `url` fields in the `SUBJECTS` list with fresh direct links if a specific portrait fails to render. A fallback base64 string is included for the primary subject to ensure basic functionality.

In [ ]:
import os
import requests
import numpy as np
import cv2
import imageio
from scipy.spatial import cKDTree
from PIL import Image, ImageDraw, ImageFont
from tqdm.notebook import tqdm
from base64 import b64encode, b64decode
from IPython.display import HTML
from google.colab import files
import io

# --- Configuration and Assets ---

# List of subjects to be animated with their descriptions and image sources
SUBJECTS = [
    {"name": "Dedan Kimathi", "desc": "Freedom Fighter", "url": "https://qiraatafrican.com/en/wp-content/uploads/2023/11/Dedan-Kimathi-Waciuri-was-hanged-and-buried-in-an-unmarked-grave-at-Kamiti-Maximum-Prison-in-Nairobi.jpg"},
    {"name": "Patrice Lumumba", "desc": "Pan-Africanist", "url": "https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcSgrycYwqKqFFm3Zondh9iHYrfNUHnnCI2odxH7YCTotFndg-vV7JpHy2BVwnJdveMJBrZM91dzEBOMPebF8FdKfQmnlYxPi4-q6GgTmNo&s=10"},
    {"name": "Bob Marley", "desc": "Musical Legend", "url": "https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcRchzjqMqrYlNKBAlVazeSt4yLPjY8Lj1Is-UUt1Voe0PxeNNRvG18XUQO2tx4f7F8dohjq9wKxVjiI3DU3k8JE_znJyUDAi2FYDPbslX8&s=10"},
    {"name": "IShowSpeed", "desc": "Digital Icon", "url": "https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcTTzk9t50otMN0dyLyctvTfih17dQuPbWNFu3nhuM1lfEEGMuMuH2Pg1Y4wOUAfEpdubbQGl9zxg8w5jG5H90KblSTdTdhaAndalkb1hmc&s=10"}
]

# Fallback base64 string for Dedan Kimathi to ensure robustness against broken links
KIMATHI_B64 = "/9j/4AAQSkZJRgABAQAAAQABAAD/2wCEAAkGBxETEhUSEhIVFRUWFRUVFRUVFhUVFRUVFhUWFxYWFRUYHSggGBolGxUVITEhJSkrLi8uFx8zODMtNygtLisBCgoKDQ0FDg8OECsZExkrKysrKysrKzcrKysrKysrKysrKysrKysrKysrKysrKysrKysrKysrKysrKysrKysrK//AABEIAPsAyQMBIgACEQEDEQH/xAAcAAABBQEBAQAAAAAAAAAAAAACAQMEBQYABwj/xABGEAACAQIEAwYDBQYDBAsBAAABAgADEQQFEiEGMUETIlFhcYEHMpFCobHB8BQjUnLR4WKS8RczQ4IWJTREU2OTorLC0hX/xAAUAQEAAAAAAAAAAAAAAAAAAAAA/8QAFBEBAAAAAAAAAAAAAAAAAAAAAP/aAAwDAQACEQMRAD8Ag0FHtykpKQvEo0jbaSqdMi14B0FtJI6C0FKMf7PlAWioHSTKVOM9nykqkIEimm0eFG8Ggsl0xAYTDgSQqQtMNFgEqwws4CPIIDZSCyRcVjaNNkWpVRGqNppqzAM7eCA7sdxyj5SBHAh2hlZwEAQsUiOhYhEBgiCRHWEQrAaiERzTO0wG7QlEMLCAgMssHTHTEgebYVDf1lkq+MYwlO8saVPxgCtIw1TcSSKe06nStz8YBIsfWlFCR9EgLRSSkEaUR5YB6I4giCGDAK0rMRxHg6dLt3xFMUrkagwN2KsB+ygN3XhyE8h+pLLj7iwYCiNK669W60UH8VvmPkLj1njmXcGYqu2uqQmoljcXN2NybDYc4F1R+Ixp5hXxhoCuHHZ0dZ0NSpKxtosptqvczWZN8W6RRRiaLCpY3aibqeZ+V9OnptczKf7NGF71vTbeVGY8G4miCRaoB4bN/lPP2ge8ZDxDhcYpbD1Q9jZl3VwbX3U78uo2lqFnyhRrtTqCpTZkqIQQykqysPA8wZ6b8P/iTWFTssYxqKyKlN7gMGUsd7/M7BreZAgexmIRG8NiA6BwrKDewYWNgSL28Da48iIZaAMFpwnGAM60SEsAlE4idedABhEtDMSBhsIpliiSPhUtJemASiSFW8bopePqIChY6giRxTAVRHVEFBH1WASiZfjzjCll9Lo9dwRTpAi42+dx0UffO494lbA0i5wzVaToyM6uF0VG2VW6gEX3HhPn7Al69dQ7FmYi7MSxsPNiTYcoG74YwtSuxxOKZqlVze7G+hSdlUH5R5Cek5atMfMOkyGFxK0u8V7gGwA7zcrkCXWV8aUGqdk+FqIL2DMNze25Xp/aBpMSaZUEC+3vKPNdJUix9b2I95f5njqNCn2hXa2wA3O3ITFLxW2IZh+ystMcnFwPVvL74GA4lwFy1VRZgbP5221esoFFxY9bzXcTVSrFhYo21wQRe36+kxhP3XgfRXw4z1sXgKbO16tO9KqdrkpbSxA8VKn6zSsZ4h8Gc37HGNQOsriEChVBYdqpurG3Iaddz6T2+okAbziYE68BbxVgxYDlp0RYsATEhEQYGXpLJtOn4xqkkl0hAXR4TrR8LEYQBtDQTtEJIDiCPqIykyPxV4gbCYRTSqvTrPUUUylum7argjTbp4kQPMvixmGO/bKmHxFUGkG10qVNgUVNwmpR9uwub77+FpnOFlviqY9fwMh4qpUqO1SqxZ2JZmY3JJ6mW/BmEL4kdLKTf1gbVcZiKdVagw71mFigDCnTCjYDHe+pLLj7iwYCiNK669W60UH8VvmPkLj1njmXcGYqu2uqQmoljcXN2NybDYc4F1R+Ixp5hXxhoCuHHZ0dZ0NSpKxtosptqvczWZN8W6RRRiaLCpY3aibqeZ+V9OnptczKf7NGF71vTbeVGY8G4miCRaoB4bN/lPP2ge8ZDxDhcYpbD1Q9jZl3VwbX3U78uo2lqFnyhRrtTqCpTZkqIQQykqysPA8wZ6b8P/iTWFTssYxqKyKlN7gMGUsd7/M7BreZAgexmIRG8NiA6BwrKDewYWNgSL28Da48iIZaAMFpwnGAM60SEsAlE4idedABhEtDMSBhsIpliiSPhUtJemASiSFW8bopePqIChY6giRxTAVRHVEFBH1WASiZfjzjCll9Lo9dwRTpAi42+dx0UffO494lbA0i5wzVaToyM6uF0VG2VW6gEX3HhPn7Al69dQ7FmYi7MSxsPNiTYcoG74YwtSuxxOKZqlVze7G+hSdlUH5R5Cek5atMfMOkyGFxK0u8V7gGwA7zcrkCXWV8aUGqdk+FqIL2DMNze25Xp/aBpMSaZUEC+3vKPNdJUix9b2I95f5njqNCn2hXa2wA3O3ITFLxW2IZh+ystMcnFwPVvL74GA4lwFy1VRZgbP5221esoFFxY9bzXcTVSrFhYo21wQRe36+kxhP3XgfRXw4z1sXgKbO16tO9KqdrkpbSxA8VKn6zSsZ4h8Gc37HGNQOsriEChVBYdqpurG3Iaddz6T2+okAbziYE68BbxVgxYDlp0RYsATEhEQYGXpLJtOn4xqkkl0hAXR4TrR8LEYQBtDQTtEJIDiCPqIykyPxV4gbCYRTSqvTrPUUUylum7argjTbp4kQPMvixmGO/bKmHxFUGkG10qVNgUVNwmpR9uwub77+FpnOFlviqY9fwMh4qpUqO1SqxZ2JZmY3JJ6mW/BmEL4kdLKTf1gbVcZiKdVagw71mFigDCnTCjYD"

# Animation parameters
WATERMARK = "Mugambi Ndwiga | @craftsandengineering"
OUTPUT_FILE = "stipple_portraits.mp4"
SIZE = 512           # Video resolution (512x512)
N_POINTS = 4000      # Target number of stipple points
ITERATIONS = 40      # Number of Lloyd's relaxation steps
FPS = 30             # Frames per second

def download_image(url, target_size=(SIZE, SIZE)):
    """
    Downloads an image from a URL or decodes it from base64, then cleans it.
    Args:
        url (str): Image URL or 'base64_kimathi' keyword.
        target_size (tuple): Desired (width, height).
    Returns:
        np.ndarray: Grayscale preprocessed image.
    """
    if url == "base64_kimathi":
        b64_str = KIMATHI_B64.strip()
        # Ensure correct base64 padding
        missing_padding = len(b64_str) % 4
        if missing_padding:
            b64_str += "=" * (4 - missing_padding)
        img_data = b64decode(b64_str)
    else:
        # Set headers to avoid HTTP 403/429 from image hosts
        headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'}
        resp = requests.get(url, headers=headers, timeout=15)
        if resp.status_code != 200:
             if "qiraatafrican.com" in url:
                 return download_image("base64_kimathi", target_size)
             raise ValueError(f"Status {resp.status_code} for {url}")
        img_data = resp.content

    # Convert to grayscale and resize
    img_array = np.asarray(bytearray(img_data), dtype="uint8")
    img = cv2.imdecode(img_array, cv2.IMREAD_GRAYSCALE)
    if img is None: raise ValueError("Decode failed")
    img = cv2.resize(img, target_size, interpolation=cv2.INTER_LANCZOS4)
    img = cv2.GaussianBlur(img, (5, 5), 1)
    return img

def get_importance_map(gray_img):
    """
    Creates a probability map for point placement based on image density.
    Args:
        gray_img (np.ndarray): The input grayscale image.
    Returns:
        tuple: (weights_matrix, flattened_probability_map)
    """
    inverted = 255.0 - gray_img.astype(np.float32)
    inverted /= 255.0
    # Apply contrast power factor (p=1.8)
    weights = np.power(inverted, 1.8)
    prob_map = weights.flatten() / weights.sum()
    return weights, prob_map

def lloyd_relaxation(weights, prob_map, n_points=N_POINTS, k_iters=ITERATIONS):
    """
    Refines point positions using a weighted Voronoi Centroidal approach.
    Args:
        weights (np.ndarray): Importance weights.
        prob_map (np.ndarray): Probability distribution.
    Returns:
        list: Snapshots of point positions over iterations.
    """
    h, w = weights.shape
    # Initial random placement based on probability map
    idx = np.random.choice(h * w, size=n_points, replace=False, p=prob_map)
    y, x = np.unravel_index(idx, (h, w))
    points = np.stack([x, y], axis=1).astype(np.float32)

    yy, xx = np.mgrid[:h, :w]
    pixel_coords = np.stack([xx.ravel(), yy.ravel()], axis=1)

    snapshots = []
    for i in range(k_iters):
        snapshots.append(points.copy())
        # Vectorized Lloyd's step using k-d tree and bincounts
        tree = cKDTree(points)
        _, labels = tree.query(pixel_coords)
        w_flat = weights.ravel()
        sum_w = np.bincount(labels, weights=w_flat, minlength=n_points)
        sum_x = np.bincount(labels, weights=w_flat * pixel_coords[:, 0], minlength=n_points)
        sum_y = np.bincount(labels, weights=w_flat * pixel_coords[:, 1], minlength=n_points)

        nz = sum_w > 0
        points[nz, 0] = sum_x[nz] / sum_w[nz]
        points[nz, 1] = sum_y[nz] / sum_w[nz]

        # Safety: re-randomize points that lost their Voronoi cell
        if not np.all(nz):
            r_idx = np.random.choice(h*w, size=np.sum(~nz), p=prob_map)
            ry, rx = np.unravel_index(r_idx, (h, w))
            points[~nz] = np.stack([rx, ry], axis=1)

    snapshots.append(points)
    return snapshots

def render_stipple_frame(points, weights, size=SIZE):
    """
    Draws variable-sized circles on a canvas based on local image density.
    """
    canvas = np.full((size, size, 3), (240, 245, 245), dtype=np.uint8)
    for i in range(len(points)):
        px, py = np.clip(points[i].astype(int), 0, size-1)
        # Radius scales between 1.0 and 3.5
        radius = 1.0 + (2.5 * weights[py, px])
        cv2.circle(canvas, (px, py), int(round(radius)), (26, 26, 26), -1, cv2.LINE_AA)
    return canvas

def add_overlay(img_np, watermark_text, lower_third=None):
    """
    Adds text watermark and lower-third subject descriptions.
    """
    img = Image.fromarray(img_np)
    draw = ImageDraw.Draw(img, "RGBA")
    f = ImageFont.load_default()

    # Burn-in watermark
    tw, th = draw.textbbox((0,0), watermark_text, font=f)[2:]
    draw.text((SIZE - tw - 10, SIZE - th - 10), watermark_text, fill=(255, 255, 255, 150), font=f)

    # Add descriptive label
    if lower_third:
        draw.rectangle([0, SIZE-60, SIZE, SIZE-20], fill=(0, 0, 0, 100))
        draw.text((20, SIZE-50), lower_third, fill=(255, 255, 255, 255), font=f)
    return np.array(img)

# --- Main Animation Loop ---

writer = imageio.get_writer(OUTPUT_FILE, fps=FPS, codec='libx264', quality=8)
print("Generating Frames...")

# Sequence 1: Opening Title Card (2 seconds)
for _ in range(60):
    c = np.zeros((SIZE, SIZE, 3), dtype=np.uint8)
    img = Image.fromarray(c); draw = ImageDraw.Draw(img)
    draw.text((120, 230), "Voronoi Stippling Portraits", fill="white")
    writer.append_data(add_overlay(np.array(img), WATERMARK))

# Sequence 2: Processing Subjects
for sub in tqdm(SUBJECTS):
    try:
        img_gray = download_image(sub['url'])
        weights, p_map = get_importance_map(img_gray)
        snapshots = lloyd_relaxation(weights, p_map)

        # Each subject gets 7.5 seconds (225 frames)
        for f in range(225):
            lbl = None
            if f < 150:
                # Transition: Lloyd refinement and point growth (500 to 4000)
                s_idx = int((f / 150) * ITERATIONS)
                cnt = int(np.interp(f, [0, 30], [500, N_POINTS])) if f < 30 else N_POINTS
                frame = render_stipple_frame(snapshots[s_idx][:cnt], weights)
            else:
                # Hold: Display final portrait with name
                frame = render_stipple_frame(snapshots[-1], weights)
                if f < 195: lbl = f"{sub['name']} · {sub['desc']}"
            writer.append_data(add_overlay(frame, WATERMARK, lbl))
    except Exception as e: print(f"Error with {sub['name']}: {e}")

# Sequence 3: Closing Credits (1 second)
for _ in range(30):
    c = np.zeros((SIZE, SIZE, 3), dtype=np.uint8)
    img = Image.fromarray(c); draw = ImageDraw.Draw(img)
    draw.text((160, 250), "Art by Mugambi Ndwiga", fill="white")
    writer.append_data(add_overlay(np.array(img), WATERMARK))

writer.close()

# --- Display and Export ---
mp4 = open(OUTPUT_FILE,'rb').read()
durl = "data:video/mp4;base64," + b64encode(mp4).decode()
display(HTML(f'<video width="512" controls><source src="{durl}" type="video/mp4"></video>'))
files.download(OUTPUT_FILE)